In [1]:
# I didn't want to have to fuck with the import shit so I just did this to save time lmao
import os
import json
from typing import List, Dict, Any
import re

def parse_book(path: str) -> List[Dict[str, Any]]:
    """
    Parses the book into chapters and sections.
    Save the parsed book to a JSON file in the output directory.

    Args:
        path: Path to the book file.
    """

    # Check if the book file exists
    if not os.path.exists(path):
        raise FileNotFoundError(f"book file not found at {path}")

    print(f"Parsing book {path}...")

    # Read the book file
    with open(path, "r", encoding="utf-8") as f:
        file_content = f.read()

    # PARSE THE BOOK BY CHAPTERS AND SECTIONS
    parsed_book = []
    ch_blocks = re.split(r"^#CHAPTER\s*", file_content, flags=re.MULTILINE)
    for ch_idx, ch_block in enumerate(ch_blocks):
        if not ch_block:
            continue

        # Separate chapter title from text
        lines = ch_block.splitlines()
        ch_title = lines[0].strip()
        ch_text = "\n".join(lines[1:])

        # Split sections within the chapter
        sections = []
        sec_blocks = re.split(r"^#SECTION\s*", ch_text, flags=re.MULTILINE)

        # Case 1: No #SECTION in chapter. Treat entire chapter text as a single, untitled section
        if len(sec_blocks) == 1:
            chapter_description = sec_blocks[0].strip()
            if not chapter_description:
                chapter_description = "<No text in section>"

            sections.append(
                {
                    "section_idx": 0,
                    "section_title": "<Chapter description>",
                    "section_text": chapter_description,
                }
            )
        else:
            # Case 2: There are sections. The first block may be the chapter description
            running_idx = 0
            chapter_description = sec_blocks[0].strip()
            if chapter_description:
                sections.append(
                    {
                        "section_idx": running_idx,
                        "section_title": "<Chapter description>",
                        "section_text": chapter_description,
                    }
                )
                running_idx += 1

            # Parse each real section: first non-empty line is the title, rest is content
            for sec_block in sec_blocks[1:]:
                block = sec_block.strip()
                if not block:
                    continue

                sec_lines = sec_block.splitlines()
                sec_title = sec_lines[0].strip()
                sec_text = "\n".join(sec_lines[1:]).strip()
                if not sec_text:
                    sec_text = "<No text in section>"

                sections.append(
                    {
                        "section_idx": running_idx,
                        "section_title": sec_title,
                        "section_text": sec_text,
                    }
                )
                running_idx += 1

        # Add the chapter to the parsed book
        parsed_book.append(
            {
                "chapter_idx": ch_idx,
                "chapter_title": ch_title,
                "sections": sections,
            }
        )

    return parsed_book

In [2]:
# 1. parse document using the tagging/parse_books.py
parsed_book = parse_book("../books/《人紀傷寒論》.txt")

Parsing book ../books/《人紀傷寒論》.txt...


In [10]:
# Tokenize each document
import jieba

corpus = []
for chapter in parsed_book:
    for section in chapter["sections"]:
        corpus.append(section["section_text"])

tokenized_corpus = [list(jieba.cut(doc)) for doc in corpus]

In [11]:
# Create BM25 index
from rank_bm25 import BM25Okapi

bm25 = BM25Okapi(tokenized_corpus)

In [12]:
queries = [
  "病人，女，35歲，主訴頭痛三個月，伴有眩暈、失眠。頭痛以太陽穴為主，每日下午加重，月經不調，量少色暗。舌質暗紅，苔薄白，脈弦細。",
  "病人，男，45歲，慢性胃炎病史5年，近期胃脘脹痛，噯氣頻繁，食慾不振。疼痛在飯後加重，喜溫喜按，大便溏薄，舌淡苔白膩，脈沉緩。",
  "病人，女，28歲，產後2個月，惡露不盡，小腹疼痛，按之痛甚。伴有低熱，口乾不欲飲，舌質紫暗有瘀斑，脈澀。",
  "病人，男，60歲，咳嗽痰多三週，痰色白稠，胸悶氣短，動則加重。既往有慢性支氣管炎病史，舌淡苔白滑，脈滑。",
  "病人，女，50歲，更年期綜合征，潮熱盜汗，心煩易怒，失眠多夢。月經紊亂，腰膝酸軟，舌紅少苔，脈細數。"
]

In [13]:
# Run BM25 for given query
tokenized_queries = [list(jieba.cut(query)) for query in queries]

In [19]:
# Get top matches
for i, tokenized_query in enumerate(tokenized_queries):
    scores = bm25.get_scores(tokenized_query)
    ranked = sorted(zip(scores, corpus), reverse=True)
    
    # save to a file
    file_path = f'output/query_{i}.txt'
    with open(file_path, 'w') as f:
        f.write(f'query {i}\n')
        f.write(f'query: {queries[i]}\n')
        f.write('----------\n')
        for i, (score, doc) in enumerate(ranked[:25]):
            f.write(f"rank: {i}\n")
            f.write(f"score: {score:.3f}\n")
            f.write(f"text: {doc}\n")
            f.write('=====\n')
        f.write('\n\n')